# 🌳 Binary Heaps & Heap Sort
**Seneca Polytechnic — DSA Study Notes**

---

## 📋 What We'll Cover
1. Priority Queues — the *why*
2. Binary Heap — definitions
3. Insertion (percolate up)
4. Removal (percolate down)
5. Array-based implementation
6. Heapify
7. Heap Sort
8. Complexity summary

---
## 1. Priority Queues — The *Why*

A **priority queue** is like a regular queue, but instead of *first-in, first-out*, the item that comes out next is the one with the **highest priority**.

Think of a hospital emergency room — a patient with a heart attack gets seen before someone with a sprained ankle, regardless of arrival order.

### Naive implementation with a sorted list
If you just keep a sorted list:
- **Insert** → O(n) (you have to find the right spot)
- **Remove** → O(n) (shifting elements)

This is bad. A sort built on top of this wouldn't be better than simple O(n²) sorts.

> **Key insight:** We only *ever* care about the highest-priority item. Everything else can be wherever it wants.

---
## 2. Binary Heap — Basic Definitions

### Binary Tree
A tree where every node has **at most 2 children** (left and right).

### Complete Binary Tree
A binary tree where:
- Every level is fully filled **except possibly the last**
- The last level is filled **left to right** (no gaps skipping to the right)

```
✅ Complete:         ❌ Not complete (gap before existing node):
      1                      1
     / \                    / \
    2   3                  2   3
   / \  /                 /     \
  4   5 6                4       5
```

### Heap Order Property
For every node: **parent has higher priority than its children**.

- **Min-Heap** → smaller value = higher priority → root is the *minimum*
- **Max-Heap** → larger value = higher priority → root is the *maximum*

```
Min-Heap example:        Max-Heap example:
        2                        9
       / \                      / \
      5   3                    7   8
     / \ / \                  / \ / \
    6  16 17 13              4  5 3  1
```

> **A Binary Heap = Complete Binary Tree + Heap Order Property**

---
## 3. Insertion — Percolate Up

To insert a value while keeping the heap valid:

1. Place the new value at the **leftmost open spot on the bottom level** (maintains complete tree)
2. Compare it with its **parent**
3. If it violates the heap order property → **swap with parent**
4. Repeat until the value is in the correct position

This upward movement is called **percolate up** (or bubble up).

**Time complexity:** O(log n) — at most log n swaps (height of the tree)

### Worked Example: Build a Min-Heap by inserting [10, 6, 20, 5, 16, 17, 13, 2]

```
Insert 10:         Insert 6:         Insert 20:
   10                 6                  6
                     /                  / \
                    10                10   20

Insert 5:          Insert 16:        Insert 17:
   5                  5                 5
  / \                / \              / \
 6   20             6   20           6   17
/                  / \              / \ /
10                10  16           10 16 20

Insert 13:         Insert 2 (percolates all the way up!):
   5                  2
  / \                / \
 6   13             5   13
/ \ / \            / \ / \
10 16 17 20       6  16 17 20
                 /
                10
```

In [ ]:
class MinHeap:
    def __init__(self):
        self.heap = []

    def insert(self, val):
        self.heap.append(val)       # Step 1: add at end (leftmost open spot)
        self._percolate_up(len(self.heap) - 1)

    def _percolate_up(self, i):
        while i > 0:
            parent = (i - 1) // 2
            if self.heap[i] < self.heap[parent]:   # min-heap: smaller = higher priority
                self.heap[i], self.heap[parent] = self.heap[parent], self.heap[i]
                i = parent
            else:
                break

    def peek(self):
        return self.heap[0] if self.heap else None

    def __repr__(self):
        return str(self.heap)


# Demo — insert [10, 6, 20, 5, 16, 17, 13, 2]
h = MinHeap()
for v in [10, 6, 20, 5, 16, 17, 13, 2]:
    h.insert(v)
    print(f"After inserting {v:2d}: {h}")

---
## 4. Removal — Percolate Down

We always remove the **root** (highest priority item). But we can't just delete it — the tree would break.

**Steps:**
1. Save the root value (to return it)
2. Move the **last element** to the root (maintains complete tree shape)
3. Remove the last element
4. **Percolate down:** compare root with its children, swap with the **higher priority child** if needed
5. Repeat until the moved value is in the right spot

**Time complexity:** O(log n)

### Example: Remove from min-heap

```
Start:                Move last to root:     Percolate down:
       2                    10                    5
      / \                  / \                  / \
     5   13               5   13               6   13
    / \ / \              / \ /                / \ /
   6  16 17 20          6  16 17             10  16 17
  /
 10
```
> Removed: 2

In [ ]:
class MinHeap:
    def __init__(self):
        self.heap = []

    def insert(self, val):
        self.heap.append(val)
        self._percolate_up(len(self.heap) - 1)

    def delete_min(self):
        if not self.heap:
            return None
        if len(self.heap) == 1:
            return self.heap.pop()

        root = self.heap[0]             # Save the min
        self.heap[0] = self.heap.pop()  # Move last to root
        self._percolate_down(0)         # Fix heap order
        return root

    def _percolate_up(self, i):
        while i > 0:
            parent = (i - 1) // 2
            if self.heap[i] < self.heap[parent]:
                self.heap[i], self.heap[parent] = self.heap[parent], self.heap[i]
                i = parent
            else:
                break

    def _percolate_down(self, i):
        n = len(self.heap)
        while True:
            left  = 2 * i + 1
            right = 2 * i + 2
            smallest = i

            if left < n and self.heap[left] < self.heap[smallest]:
                smallest = left
            if right < n and self.heap[right] < self.heap[smallest]:
                smallest = right

            if smallest != i:
                self.heap[i], self.heap[smallest] = self.heap[smallest], self.heap[i]
                i = smallest
            else:
                break

    def peek(self):
        return self.heap[0] if self.heap else None

    def __repr__(self):
        return str(self.heap)


# Build and then repeatedly remove
h = MinHeap()
for v in [10, 6, 20, 5, 16, 17, 13, 2]:
    h.insert(v)

print("Heap:", h)
print()
print("Removing elements (should come out sorted):")
while h.heap:
    print(f"  removed: {h.delete_min()}  | heap now: {h}")

---
## 5. Array-Based Implementation

The heap doesn't need an actual tree with node objects. A simple **array** works perfectly because the heap is a *complete* binary tree.

### Index Math

For a node at index `i` (0-indexed):

| Relationship | Formula |
|---|---|
| Left child   | `2*i + 1` |
| Right child  | `2*i + 2` |
| Parent       | `(i-1) // 2` |
| First non-leaf | `(n-2) // 2` |

### Visualising the mapping

```
Tree:                Array:
       2             [2, 5, 13, 6, 16, 17, 20, 10]
      / \             0  1   2  3   4   5   6   7
     5   13
    / \ / \
   6  16 17 20
  /
 10

Node at index 1 (value=5):
  parent  = (1-1)//2 = 0  → value 2  ✅
  left    = 2*1+1   = 3  → value 6  ✅
  right   = 2*1+2   = 4  → value 16 ✅
```

In [ ]:
# Quick sanity check of the index math
arr = [2, 5, 13, 6, 16, 17, 20, 10]

for i, val in enumerate(arr):
    parent = arr[(i-1)//2] if i > 0 else None
    left   = arr[2*i+1]    if 2*i+1 < len(arr) else None
    right  = arr[2*i+2]    if 2*i+2 < len(arr) else None
    print(f"arr[{i}]={val} → parent={parent}, left={left}, right={right}")

---
## 6. Heapify — Building a Heap in Place

Given an *unsorted* array, we want to turn it into a heap **without extra space**.

### Naive way (don't do this for performance)
Insert elements one by one → O(n log n)

### Heapify (the fast way) → O(n)

**Key observation:** All **leaf nodes** are already valid heaps (no children → can't violate anything).

So we only need to fix the **non-leaf nodes**, starting from the last one and working toward the root.

**Algorithm:**
1. Find the last non-leaf: index `(n-2) // 2`
2. For each node from that index down to 0 → call `percolate_down(i)`

### Example: Build max-heap from [1, 6, 3, 5, 8, 4, 7]

```
Initial array:         As a tree:
[1, 6, 3, 5, 8, 4, 7]       1
                            / \
                           6   3
                          / \ / \
                         5  8 4  7

Leaves (green) = indices 3,4,5,6 → skip them
First non-leaf = (7-2)//2 = 2 (value=3)

heapify(2): node=3, children=4,7 → swap with 7
       1                1
      / \              / \
     6   3    →       6   7
    / \ / \          / \ / \
   5  8 4  7        5  8 4  3

heapify(1): node=6, children=8,5 → swap with 8
       1                1
      / \              / \
     6   7    →       8   7
    / \ / \          / \ / \
   5  8 4  3        5  6 4  3

heapify(0): node=1, children=8,7 → swap with 8
            then node=1 at index 1, children=5,6 → swap with 6
       1                8
      / \              / \
     8   7    →       6   7
    / \ / \          / \ / \
   5  6 4  3        5  1 4  3

Final max-heap: [8, 6, 7, 5, 1, 4, 3]
```

In [ ]:
def heapify_down(arr, n, i, is_max=True):
    """Percolate arr[i] down to its correct position in a heap of size n."""
    target = i
    left   = 2 * i + 1
    right  = 2 * i + 2

    if is_max:
        if left  < n and arr[left]  > arr[target]: target = left
        if right < n and arr[right] > arr[target]: target = right
    else:
        if left  < n and arr[left]  < arr[target]: target = left
        if right < n and arr[right] < arr[target]: target = right

    if target != i:
        arr[i], arr[target] = arr[target], arr[i]
        heapify_down(arr, n, target, is_max)


def build_heap(arr, is_max=True):
    n = len(arr)
    start = (n - 2) // 2   # last non-leaf
    print(f"Array: {arr}")
    print(f"Last non-leaf index: {start} (value={arr[start]})")
    print()
    for i in range(start, -1, -1):
        heapify_down(arr, n, i, is_max)
        print(f"After heapify({i}): {arr}")
    return arr


data = [1, 6, 3, 5, 8, 4, 7]
print("=== Building Max-Heap ===")
result = build_heap(data[:])
print(f"\nFinal max-heap: {result}")

---
## 7. Heap Sort

Heap sort works in **two phases**:

### Phase 1 — Build a max-heap in place: O(n)
Using the Heapify algorithm from Section 6.

### Phase 2 — Repeatedly extract the max: O(n log n)
- Swap the root (max) with the last element
- Shrink the heap by 1 (that element is now sorted at the back)
- Percolate down the new root
- Repeat until the heap has 1 element

After all extractions, the array is **sorted ascending** (largest values settled at the back).

```
Max-heap: [8, 6, 7, 5, 1, 4, 3]

Step 1: swap root(8) with last(3) → [3, 6, 7, 5, 1, 4, | 8]
        heapify(0) on size 6      → [7, 6, 4, 5, 1, 3, | 8]

Step 2: swap root(7) with last(3) → [3, 6, 4, 5, 1, | 7, 8]
        heapify(0) on size 5      → [6, 5, 4, 3, 1, | 7, 8]

... and so on until sorted: [1, 3, 4, 5, 6, 7, 8]
```

### Why use max-heap for ascending sort?
The largest element gets placed at the end first, then the second largest, etc. — building the sorted array from right to left.

> ⚠️ The naive version (insert into a Heap object, then delete) uses O(n) extra space — like merge sort. The in-place approach avoids that.

In [ ]:
def heap_sort(arr):
    n = len(arr)

    # Phase 1: Build max-heap in place — O(n)
    for i in range((n - 2) // 2, -1, -1):
        heapify_down(arr, n, i, is_max=True)

    print(f"Max-heap built: {arr}")
    print()

    # Phase 2: Extract max one by one — O(n log n)
    for end in range(n - 1, 0, -1):
        arr[0], arr[end] = arr[end], arr[0]     # swap max to end
        heapify_down(arr, end, 0, is_max=True)  # fix heap (exclude sorted part)
        print(f"  sorted tail: {arr[end:]}   heap: {arr[:end]}")

    return arr


# heapify_down defined in previous cell — reusing it here
data = [5, 3, 8, 1, 9, 2, 7, 4, 6]
print(f"Input:  {data}")
result = heap_sort(data[:])
print(f"\nSorted: {result}")

---
## 8. Complexity Summary

| Operation | Time | Why |
|---|---|---|
| Insert | O(log n) | Percolate up at most height of tree |
| Delete (remove root) | O(log n) | Percolate down at most height of tree |
| Build heap (n insertions) | O(n log n) | n × O(log n) |
| Heapify (in-place) | **O(n)** | Most nodes are near leaves (short paths) |
| Heap Sort — Phase 1 | O(n) | Heapify |
| Heap Sort — Phase 2 | O(n log n) | n deletions × O(log n) |
| **Heap Sort total** | **O(n log n)** | |

### Space complexity
- In-place heap sort: **O(1)** extra space (array only)
- Naive heap sort (with heap object): **O(n)** extra space

### Why is Heapify O(n) and not O(n log n)?
Most nodes are near the **bottom** of the tree, so they have very short percolate-down paths.
Only a few nodes near the top have a long path. The math works out to O(n) total work.

In [ ]:
# Final: complete self-contained MinHeap class
class MinHeap:
    """Min-heap using a dynamic array."""

    def __init__(self):
        self._data = []

    # ---------- public API ----------

    def insert(self, val):
        self._data.append(val)
        self._up(len(self._data) - 1)

    def delete_min(self):
        if not self._data:
            raise IndexError("Heap is empty")
        if len(self._data) == 1:
            return self._data.pop()
        root = self._data[0]
        self._data[0] = self._data.pop()
        self._down(0)
        return root

    def peek(self):
        return self._data[0] if self._data else None

    def __len__(self):  return len(self._data)
    def __repr__(self): return f"MinHeap({self._data})"

    # ---------- internal helpers ----------

    def _up(self, i):
        while i > 0:
            p = (i - 1) // 2
            if self._data[i] < self._data[p]:
                self._data[i], self._data[p] = self._data[p], self._data[i]
                i = p
            else:
                break

    def _down(self, i):
        n = len(self._data)
        while True:
            lo = i
            l, r = 2*i+1, 2*i+2
            if l < n and self._data[l] < self._data[lo]: lo = l
            if r < n and self._data[r] < self._data[lo]: lo = r
            if lo == i: break
            self._data[i], self._data[lo] = self._data[lo], self._data[i]
            i = lo


# --- tests ---
import random
random.seed(42)
vals = random.sample(range(1, 51), 10)
print("Input values:", vals)

h = MinHeap()
for v in vals:
    h.insert(v)
print("Heap array: ", h)

sorted_out = []
while len(h):
    sorted_out.append(h.delete_min())
print("Sorted output:", sorted_out)
assert sorted_out == sorted(vals), "❌ Something went wrong!"
print("✅ MinHeap works correctly!")

---
## 📝 Quick Reference Cheatsheet

```
HEAP RULES
──────────
• Complete binary tree (filled left to right)
• Parent always has higher priority than children
• Min-heap: parent ≤ children  →  root = minimum
• Max-heap: parent ≥ children  →  root = maximum

ARRAY INDEX MATH (0-indexed)
────────────────────────────
  parent(i)      = (i-1) // 2
  left_child(i)  = 2*i + 1
  right_child(i) = 2*i + 2
  first_leaf     = (n-1) // 2 + 1   (or just > (n-2)//2)

OPERATIONS
──────────
  insert      → add at end, percolate UP      O(log n)
  delete_min  → swap root↔last, percolate DOWN O(log n)
  build_heap  → heapify from last non-leaf     O(n)

HEAP SORT (ascending, uses MAX-HEAP)
────────────────────────────────────
  1. Build max-heap in place        O(n)
  2. Swap root with last, shrink,   O(n log n)
     percolate down — repeat
  Total: O(n log n),  Space: O(1)
```